# Chroma: векторная база данных для научных коллекций

**Проект «ИИ для учёных».** Практический блокнот к разделу «Инструменты».

## Что делает этот инструмент

Chroma — открытая векторная база данных: она хранит числовые представления текстов (эмбеддинги) вместе с произвольными метаданными и выдаёт наиболее похожие на запрос записи за миллисекунды. Это строительный блок систем RAG (Retrieval-Augmented Generation): вместо того чтобы скармливать языковой модели всю библиотеку статей, вы храните их в Chroma и по каждому вопросу извлекаете только несколько релевантных фрагментов.

Для исследователя это означает: тысячи аннотаций, протоколов или заметок можно индексировать один раз, а затем мгновенно искать по смыслу — и сужать выдачу фильтрами по году, области, автору или любому другому полю.

**Чем этот блокнот отличается от блокнотов по эмбеддингам и LLM:** построение эмбеддингов и семантический поиск уже разобраны в блокнотах «Текстовые эмбеддинги» и «Большие языковые модели». Здесь акцент на Chroma как инструменте хранения и управления: создание коллекции, добавление и обновление документов, фильтрация по метаданным, персистентное хранение на диске.

Расчётное время прохождения — около пяти минут.

## Ключевые понятия

- **Коллекция** (Collection) — именованное хранилище документов внутри базы данных Chroma, аналог таблицы в реляционной СУБД. Каждая коллекция хранит документы одного смыслового пространства.
- **Документ** — текстовый фрагмент (аннотация, абзац, заметка), который Chroma автоматически превращает в числовой вектор-эмбеддинг с помощью встроенной модели.
- **Метаданные** (metadata) — произвольные пары ключ/значение, прикреплённые к документу: область, год, автор, DOI. По ним можно фильтровать выдачу независимо от смыслового поиска.
- **Top-k запрос** (query) — поиск k ближайших по смыслу документов к текстовому запросу. Chroma возвращает документы вместе с мерой расстояния.
- **Фильтр** (`where`) — условие на метаданные, накладываемое поверх семантического поиска: «искать только статьи 2023 года из области физики».
- **PersistentClient** — режим работы, при котором коллекция сохраняется на диск и переживает перезапуск ядра. Для разовых экспериментов достаточно `EphemeralClient` (только в памяти).
- **Встроенная функция эмбеддингов** — по умолчанию Chroma использует модель `all-MiniLM-L6-v2` из библиотеки `sentence-transformers`; она скачивается автоматически, не требует API-ключей.

## 1. Установка библиотек

In [ ]:
# chromadb включает встроенную функцию эмбеддингов на базе sentence-transformers.
# Модель all-MiniLM-L6-v2 скачивается автоматически при первом запуске (~80 МБ).
# Никаких API-ключей не требуется.
!pip install -q chromadb

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`; вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Зададим небольшой корпус из 14 коротких научных аннотаций на русском языке. Каждая снабжена двумя полями метаданных: **область** (тематическая рубрика) и **год** публикации. Именно по этим полям мы будем в дальнейшем демонстрировать фильтрацию.

Корпус охватывает четыре дисциплины, что позволит наглядно показать, как поиск по смыслу находит тематически близкие статьи, а фильтр по метаданным сужает выдачу до нужного временного диапазона или области.

In [ ]:
import pandas as pd

# Корпус: короткие аннотации с метаданными.
# Поля метаданных: область (str) и год (int).
CORPUS = [
    {
        "id":       "doc_01",
        "text":     "Применение нейронных сетей для предсказания вторичной структуры белков по аминокислотной последовательности.",
        "область":  "биоинформатика",
        "год":      2022,
    },
    {
        "id":       "doc_02",
        "text":     "Сравнение алгоритмов выравнивания геномных последовательностей при анализе данных секвенирования нового поколения.",
        "область":  "биоинформатика",
        "год":      2021,
    },
    {
        "id":       "doc_03",
        "text":     "Обнаружение дифференциально экспрессированных генов методами машинного обучения в транскриптомных данных.",
        "область":  "биоинформатика",
        "год":      2023,
    },
    {
        "id":       "doc_04",
        "text":     "Прогнозирование токсичности химических соединений с помощью графовых нейронных сетей.",
        "область":  "хемоинформатика",
        "год":      2023,
    },
    {
        "id":       "doc_05",
        "text":     "Генерация молекул-кандидатов для лекарств методами глубокого обучения с подкреплением.",
        "область":  "хемоинформатика",
        "год":      2022,
    },
    {
        "id":       "doc_06",
        "text":     "Предсказание растворимости органических соединений по структурным дескрипторам молекул.",
        "область":  "хемоинформатика",
        "год":      2021,
    },
    {
        "id":       "doc_07",
        "text":     "Классификация галактик по морфологии с использованием свёрточных нейронных сетей на данных телескопа.",
        "область":  "астрофизика",
        "год":      2022,
    },
    {
        "id":       "doc_08",
        "text":     "Обнаружение гравитационных волн методами статистической обработки сигналов детектора LIGO.",
        "область":  "астрофизика",
        "год":      2023,
    },
    {
        "id":       "doc_09",
        "text":     "Определение параметров тёмной материи по данным слабого гравитационного линзирования галактик.",
        "область":  "астрофизика",
        "год":      2021,
    },
    {
        "id":       "doc_10",
        "text":     "Численное моделирование климата: оценка чувствительности модели к начальным условиям.",
        "область":  "климатология",
        "год":      2023,
    },
    {
        "id":       "doc_11",
        "text":     "Реконструкция температуры поверхности океана по спутниковым данным с помощью метода случайного леса.",
        "область":  "климатология",
        "год":      2022,
    },
    {
        "id":       "doc_12",
        "text":     "Прогнозирование экстремальных осадков с применением методов глубокого обучения на метеорологических рядах.",
        "область":  "климатология",
        "год":      2021,
    },
    {
        "id":       "doc_13",
        "text":     "Интерпретируемые модели машинного обучения для предсказания урожайности по данным дистанционного зондирования.",
        "область":  "климатология",
        "год":      2023,
    },
    {
        "id":       "doc_14",
        "text":     "Сравнительный анализ архитектур трансформеров для задач многомерной классификации в геофизике.",
        "область":  "климатология",
        "год":      2022,
    },
]

df = pd.DataFrame(CORPUS)
print(f"Документов в корпусе: {len(df)}")
print(f"Области: {sorted(df['область'].unique())}")
print(f"Годы: {sorted(df['год'].unique())}")
df[["id", "область", "год", "text"]]

## 4. Применение инструмента

### Шаг 4.1. Создание коллекции

Chroma предоставляет два режима работы:

- `EphemeralClient()` — база данных живёт только в оперативной памяти текущей сессии. Подходит для экспериментов.
- `PersistentClient(path=...)` — коллекция сохраняется на диск и доступна после перезапуска ядра. Подходит для рабочего использования.

В этом блокноте мы используем `EphemeralClient`, чтобы не зависеть от файловой системы Colab. Переключение на персистентный режим показано в шаге 4.5.

In [ ]:
import chromadb

# EphemeralClient — база данных только в памяти, без записи на диск.
# Это удобно для демонстрации; для реальной работы используйте PersistentClient.
client = chromadb.EphemeralClient()

# Создаём коллекцию с именем "science_abstracts".
# Встроенная функция эмбеддингов (DefaultEmbeddingFunction) использует
# модель all-MiniLM-L6-v2 из sentence-transformers.
# При первом вызове модель скачивается автоматически (~80 МБ).
collection = client.create_collection(
    name="science_abstracts",
    # get_or_create=True позволило бы пересоздать коллекцию без ошибки
    # при повторном запуске ячейки; здесь используем create_collection,
    # чтобы явно показать, что коллекция создаётся заново.
)

print(f"Коллекция создана: '{collection.name}'")
print(f"Документов в коллекции: {collection.count()}")

### Шаг 4.2. Добавление документов с метаданными

Метод `collection.add()` принимает три параллельных списка:

- `ids` — уникальные строковые идентификаторы (обязательно); по ним позже можно обновлять или удалять записи.
- `documents` — тексты; Chroma самостоятельно вычислит эмбеддинги.
- `metadatas` — список словарей с произвольными полями. Все значения должны быть строками, целыми числами или числами с плавающей точкой.

Добавить можно и готовые эмбеддинги через параметр `embeddings`, но встроенная функция освобождает от этой работы.

In [ ]:
# Извлекаем параллельные списки из датафрейма.
doc_ids       = df["id"].tolist()
doc_texts     = df["text"].tolist()
doc_metadatas = df[["область", "год"]].rename(columns={"область": "область", "год": "год"}).to_dict("records")

# Добавляем все документы одним вызовом.
# Chroma вычисляет эмбеддинги встроенной моделью all-MiniLM-L6-v2.
# При первом запуске модель скачивается — это займёт около минуты.
collection.add(
    ids=doc_ids,
    documents=doc_texts,
    metadatas=doc_metadatas,
)

print(f"Документов добавлено: {collection.count()}")

# Проверяем: заберём один документ по id и убедимся, что метаданные сохранились.
sample = collection.get(ids=["doc_01"])
print("\nПример — документ doc_01:")
print(f"  Текст:    {sample['documents'][0]}")
print(f"  Область:  {sample['metadatas'][0]['область']}")
print(f"  Год:      {sample['metadatas'][0]['год']}")

### Шаг 4.3. Поиск по смыслу (top-k запрос)

Метод `collection.query()` принимает список текстовых запросов и возвращает `n_results` ближайших по смыслу документов. Chroma строит эмбеддинг запроса той же моделью, что и документы, и ищет ближайших соседей по метрике L2 (или косинусного расстояния — зависит от параметра `space` при создании коллекции; по умолчанию — L2).

Обратите внимание: запрос сформулирован иначе, чем сами аннотации, — поиск идёт по смыслу, а не по совпадению слов.

In [ ]:
QUERY = "предсказание свойств молекул методами ИИ"

results = collection.query(
    query_texts=[QUERY],
    n_results=5,
    include=["documents", "metadatas", "distances"],
)

# Chroma возвращает результаты для каждого запроса в виде списков,
# поэтому берём нулевой элемент (у нас один запрос).
docs      = results["documents"][0]
metas     = results["metadatas"][0]
distances = results["distances"][0]

print(f"Запрос: «{QUERY}»")
print(f"{'Ранг':<5} {'Расстояние':<12} {'Область':<18} {'Год':<6} Текст")
print("-" * 90)
for rank, (doc, meta, dist) in enumerate(zip(docs, metas, distances), start=1):
    print(f"{rank:<5} {dist:<12.4f} {meta['область']:<18} {meta['год']:<6} {doc[:65]}...")

### Шаг 4.4. Фильтрация по метаданным (`where`)

Параметр `where` позволяет наложить жёсткое условие на поле метаданных перед семантическим поиском. Поддерживаются операторы `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, а также логические `$and` / `$or`.

В ячейках ниже мы зафиксируем запрос и сравним выдачу **без фильтра** и **с фильтром по области**. Это показывает главную практическую ценность метаданных: семантический поиск находит «похожее по смыслу», а фильтр уточняет выдачу до нужной дисциплины или временного окна.

In [ ]:
QUERY_FILTER = "глубокое обучение для анализа природных данных"

# Поиск без фильтра — вся коллекция.
res_all = collection.query(
    query_texts=[QUERY_FILTER],
    n_results=4,
    include=["documents", "metadatas", "distances"],
)

# Поиск только по климатологии.
res_climate = collection.query(
    query_texts=[QUERY_FILTER],
    n_results=4,
    where={"область": {"$eq": "климатология"}},
    include=["documents", "metadatas", "distances"],
)

# Поиск только по статьям начиная с 2022 года (включительно).
res_recent = collection.query(
    query_texts=[QUERY_FILTER],
    n_results=4,
    where={"год": {"$gte": 2022}},
    include=["documents", "metadatas", "distances"],
)


def print_results(label, res):
    docs      = res["documents"][0]
    metas     = res["metadatas"][0]
    distances = res["distances"][0]
    print(f"\n{label}")
    print(f"  {'Расстояние':<12} {'Область':<18} {'Год':<6} Текст")
    print("  " + "-" * 80)
    for doc, meta, dist in zip(docs, metas, distances):
        print(f"  {dist:<12.4f} {meta['область']:<18} {meta['год']:<6} {doc[:60]}...")


print(f"Запрос: «{QUERY_FILTER}»")
print_results("Без фильтра (вся коллекция):", res_all)
print_results("Фильтр: область = 'климатология':", res_climate)
print_results("Фильтр: год >= 2022:", res_recent)

### Шаг 4.5. Визуализация: расстояния в top-k выдаче

Построим горизонтальную диаграмму расстояний для двух сценариев: поиск без фильтра и с фильтром по области. Это показывает, как фильтр меняет состав выдачи, не трогая смысловой ранжирование внутри отфильтрованного подмножества.

Меньшее расстояние L2 соответствует большей смысловой близости к запросу.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np


def build_bar_data(res, max_label_len=52):
    """Формирует метки и расстояния из результата query()."""
    docs      = res["documents"][0]
    metas     = res["metadatas"][0]
    distances = res["distances"][0]
    labels = [
        f"{m['область']} {m['год']} | {d[:max_label_len]}{'…' if len(d) > max_label_len else ''}"
        for d, m in zip(docs, metas)
    ]
    return labels, distances


labels_all,     dists_all     = build_bar_data(res_all)
labels_climate, dists_climate = build_bar_data(res_climate)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

# --- Левая панель: без фильтра ---
y_all = np.arange(len(labels_all))
bars_all = axes[0].barh(
    y_all, dists_all,
    color=VIZ["series"][0], alpha=0.88,
)
axes[0].set_yticks(y_all)
axes[0].set_yticklabels(labels_all, fontsize=9)
axes[0].invert_yaxis()   # первый результат сверху
axes[0].set_xlabel("Расстояние L2 (меньше = ближе к запросу)")
axes[0].set_title("Без фильтра\n(все области)")
axes[0].grid(True, axis="x")

# Подписи значений
for bar, val in zip(bars_all, dists_all):
    axes[0].text(
        val + 0.005, bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}", va="center", ha="left",
        fontsize=9, color=VIZ["node_text"],
    )

# --- Правая панель: фильтр по климатологии ---
y_cl = np.arange(len(labels_climate))
bars_cl = axes[1].barh(
    y_cl, dists_climate,
    color=VIZ["series"][1], alpha=0.88,
)
axes[1].set_yticks(y_cl)
axes[1].set_yticklabels(labels_climate, fontsize=9)
axes[1].invert_yaxis()
axes[1].set_xlabel("Расстояние L2 (меньше = ближе к запросу)")
axes[1].set_title("С фильтром:\nобласть = 'климатология'")
axes[1].grid(True, axis="x")

for bar, val in zip(bars_cl, dists_climate):
    axes[1].text(
        val + 0.005, bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}", va="center", ha="left",
        fontsize=9, color=VIZ["node_text"],
    )

patch_all = mpatches.Patch(color=VIZ["series"][0], alpha=0.88, label="Все области")
patch_cl  = mpatches.Patch(color=VIZ["series"][1], alpha=0.88, label="Только климатология")
fig.legend(handles=[patch_all, patch_cl], loc="lower center",
           ncol=2, bbox_to_anchor=(0.5, -0.07), fontsize=10)

fig.suptitle(
    f"Top-k результаты для запроса: «{QUERY_FILTER}»",
    fontsize=13, y=1.02,
)
fig.tight_layout()
plt.show()

### Шаг 4.6. Обновление и удаление записей

Chroma поддерживает полноценный жизненный цикл документов:

- `collection.update()` — заменяет текст и/или метаданные существующей записи по `id`; Chroma пересчитывает эмбеддинг автоматически.
- `collection.upsert()` — добавляет запись, если `id` нет, или обновляет, если есть. Удобно при инкрементальном обновлении индекса.
- `collection.delete()` — удаляет записи по `ids` или по условию `where`.

In [ ]:
print("=== Демонстрация обновления и удаления ===\n")

# --- Обновление: исправим метаданные doc_01 (поменяем год) ---
print("До обновления:")
before = collection.get(ids=["doc_01"], include=["metadatas"])
print(f"  doc_01, год = {before['metadatas'][0]['год']}")

collection.update(
    ids=["doc_01"],
    metadatas=[{"область": "биоинформатика", "год": 2024}],
)

print("После обновления:")
after = collection.get(ids=["doc_01"], include=["metadatas"])
print(f"  doc_01, год = {after['metadatas'][0]['год']}")

# --- Upsert: добавим новую запись (если id нет) или обновим (если есть) ---
collection.upsert(
    ids=["doc_15"],
    documents=["Применение трансформеров к задачам предсказания трёхмерной структуры РНК."],
    metadatas=[{"область": "биоинформатика", "год": 2024}],
)
print(f"\nПосле upsert (doc_15) — документов в коллекции: {collection.count()}")

# --- Удаление по условию: удалим все записи 2021 года ---
collection.delete(where={"год": {"$eq": 2021}})
print(f"После удаления записей 2021 года — документов: {collection.count()}")

# Проверим: doc_02 (2021 год) больше не должен существовать.
check = collection.get(ids=["doc_02"])
print(f"doc_02 после удаления: {'не найден' if not check['ids'] else 'найден (ошибка)'}")

### Шаг 4.7. Персистентное хранение (PersistentClient)

Для реальных проектов коллекция должна переживать перезапуск ядра. Ячейка ниже показывает, как переключиться с `EphemeralClient` на `PersistentClient`. Всё остальное API остаётся идентичным.

In [ ]:
import os

# Путь к директории, в которой Chroma будет хранить файлы базы данных.
DB_PATH = "/tmp/chroma_science_db"
os.makedirs(DB_PATH, exist_ok=True)

# PersistentClient сохраняет данные на диск немедленно после каждой операции.
# После перезапуска ядра Colab достаточно снова вызвать PersistentClient(path=DB_PATH),
# и коллекция окажется доступна без повторного добавления документов.
persistent_client = chromadb.PersistentClient(path=DB_PATH)

# get_or_create_collection: если коллекция уже есть на диске — откроет её,
# если нет — создаст новую.
persistent_col = persistent_client.get_or_create_collection(
    name="science_abstracts_persistent",
)

# Добавим все документы из исходного датафрейма.
persistent_col.add(
    ids=doc_ids,
    documents=doc_texts,
    metadatas=doc_metadatas,
)

print(f"Коллекция '{persistent_col.name}' сохранена на диск: {DB_PATH}")
print(f"Документов: {persistent_col.count()}")
print()

# Имитируем «перезапуск»: открываем ту же директорию заново.
client2    = chromadb.PersistentClient(path=DB_PATH)
col2       = client2.get_or_create_collection("science_abstracts_persistent")
print(f"После 'перезапуска' — документов в коллекции: {col2.count()}")
print("Данные сохранились без повторного добавления документов.")

## 5. Интерпретация результатов

**Расстояние L2 и ранжирование.** Chroma по умолчанию использует метрику L2 (евклидово расстояние) в пространстве эмбеддингов. Чем меньше расстояние, тем выше смысловая близость документа к запросу. Абсолютные значения зависят от размерности модели и масштабирования; сравнивать имеет смысл только значения в рамках одной коллекции и одной функции эмбеддингов. Если при создании коллекции задать `metadata={"hnsw:space": "cosine"}`, Chroma перейдёт на косинусное расстояние (значения от 0 до 2, где 0 — идентично).

**Фильтр по метаданным меняет состав, но не смысл ранжирования.** Внутри отфильтрованного подмножества документы по-прежнему ранжируются по расстоянию. Это видно на диаграмме: значения расстояний у отфильтрованной выдачи могут быть выше, чем у лучшего результата без фильтра, — потому что самый близкий документ принадлежит другой области и был отсечён фильтром. Это нормально: исследователь сознательно ограничивает область поиска в обмен на тематическую релевантность.

**Когда фильтр важнее смыслового расстояния.** Если ваш запрос применим к нескольким дисциплинам (например, «машинное обучение для предсказания»), без фильтра выдача смешает области. Фильтр `where` устраняет это смешение: укажите область или год — и получите тематически чистую выдачу.

**Обновление и удаление.** После `update()` эмбеддинг пересчитывается автоматически, если изменился текст документа. Метаданные обновляются без пересчёта. Операция `delete(where=...)` позволяет массово удалять устаревшие документы по любому критерию метаданных — например, по году или DOI.

**PersistentClient.** Файлы базы данных занимают место на диске (Chroma хранит векторы и метаданные в SQLite + бинарных файлах индекса HNSW). При работе в Colab данные сохраняются в пределах одной сессии; для длительного хранения монтируйте Google Drive или используйте собственный сервер.

## 6. Попробуйте на своих данных

Замените демонстрационный корпус своими текстами. Подойдут аннотации статей, фрагменты протоколов, заметки из лабораторного журнала — любые текстовые фрагменты, по которым нужен смысловой поиск.

Минимальный список шагов:

1. Подготовьте список текстов и словарей метаданных (поля — на ваш выбор).
2. Замените `MY_TEXTS`, `MY_IDS` и `MY_METADATAS` в ячейке ниже своими данными.
3. Запустите блокнот заново от ячейки создания коллекции (раздел 4.1).
4. Задавайте запросы функцией `collection.query()` и настраивайте фильтры `where` под ваши поля метаданных.

In [ ]:
# --- Шаблон для своих данных ---
#
# Вариант 1: задать тексты вручную.
# MY_TEXTS = [
#     "Текст первого документа...",
#     "Текст второго документа...",
# ]
# MY_IDS = ["my_doc_01", "my_doc_02"]
# MY_METADATAS = [
#     {"категория": "методы", "год": 2024, "автор": "Иванов И.И."},
#     {"категория": "результаты", "год": 2023, "автор": "Петрова А.А."},
# ]
#
# Вариант 2: загрузить из CSV-файла.
# import pandas as pd
# df_my = pd.read_csv("my_documents.csv")   # колонки: id, text, категория, год
# MY_TEXTS     = df_my["text"].tolist()
# MY_IDS       = df_my["id"].astype(str).tolist()
# MY_METADATAS = df_my[["категория", "год"]].to_dict("records")
#
# Создание коллекции и добавление документов:
# my_client = chromadb.PersistentClient(path="/content/drive/MyDrive/chroma_db")
# my_col = my_client.get_or_create_collection("my_collection")
# my_col.add(ids=MY_IDS, documents=MY_TEXTS, metadatas=MY_METADATAS)
# print(f"Добавлено документов: {my_col.count()}")
#
# Поиск:
# MY_QUERY = "ваш поисковый запрос на русском"
# res = my_col.query(query_texts=[MY_QUERY], n_results=5,
#                    include=["documents", "metadatas", "distances"])
# for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
#     print(f"{dist:.4f} | {meta} | {doc[:80]}")
#
# Поиск с фильтром:
# res_filtered = my_col.query(
#     query_texts=[MY_QUERY], n_results=5,
#     where={"год": {"$gte": 2022}},   # или {"категория": {"$eq": "методы"}}
#     include=["documents", "metadatas", "distances"],
# )

print("Шаблон готов. Заполните MY_TEXTS, MY_IDS, MY_METADATAS и запустите блокнот.")

## 7. Что дальше

- Официальный сайт Chroma: https://www.trychroma.com/
- Документация Chroma: https://docs.trychroma.com/
- Расширенная фильтрация (`$and`, `$or`, `$contains`): https://docs.trychroma.com/docs/querying-collections/filtering-by-metadata
- Подключение собственных функций эмбеддингов (OpenAI, Cohere, HuggingFace, локальные модели): https://docs.trychroma.com/docs/embeddings/embedding-functions
- Интеграция с LangChain: https://python.langchain.com/docs/integrations/vectorstores/chroma/
- Интеграция с LlamaIndex: https://docs.llamaindex.ai/en/stable/examples/vector_stores/ChromaIndexDemo/
- Для коллекций от ~100 тыс. документов рассмотрите параметр индексирования HNSW (`hnsw:M`, `hnsw:ef_construction`) — он управляет компромиссом между скоростью и точностью поиска.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На диаграмме сравнения выдачи с фильтром и без него некоторые документы в отфильтрованной выдаче имеют большее расстояние L2, чем лучший результат без фильтра. Означает ли это, что фильтр «ухудшил» поиск, и как правильно интерпретировать это наблюдение?
2. Вы проиндексировали корпус из 500 аннотаций, используя встроенную модель `all-MiniLM-L6-v2`, и планируете позже добавить ещё 200 документов. Если вы захотите сменить модель эмбеддингов на другую, можно ли просто переиндексировать новые документы новой моделью, сохранив старые без изменений?
3. Chroma вернула два документа с расстояниями 0.31 и 0.34 на запрос «предсказание структуры белка». Как интерпретировать то, что разрыв между ними мал: означает ли это, что оба одинаково релевантны запросу, или нужна дополнительная оценка?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Это не ухудшение, а ожидаемое следствие ограничения области поиска. Фильтр принудительно исключает документы из других областей; внутри оставшегося подмножества ранжирование по расстоянию остаётся корректным. Исследователь сознательно принимает компромисс: ценой некоторого роста расстояния он получает тематически однородную выдачу. Абсолютные значения расстояния сравнимы только в пределах одной модели эмбеддингов и одного пространства коллекции.
2. Нет. Эмбеддинги разных моделей живут в несовместимых пространствах: векторы, полученные одной моделью, нельзя смешивать с векторами другой — метрика расстояния потеряет смысл. При смене модели необходимо переиндексировать весь корпус целиком, пересчитав все эмбеддинги новой моделью.
3. Малый разрыв означает, что по метрике данной модели документы находятся на практически одинаковом расстоянии от запроса. Это не гарантирует содержательной эквивалентности: модель `all-MiniLM-L6-v2` — компактная общая модель, оптимизированная не для узкоспециальных научных текстов. Для окончательной оценки релевантности следует прочитать оба документа или применить модель-реранкер, специализированный на научной литературе.
</details>
